In [ ]:
# <editor-fold desc="Imports">
import os
import time
import torch
import warnings
from tqdm import tqdm

import nibabel as nib
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from torch.utils.data import Dataset, DataLoader

# </editor-fold>

In [ ]:

# <editor-fold desc="Clinical Data Preparation and Cleaning">

# Default dataframe cleaning
clinical_data_path = "/scratch/tgoedietdoebe/AI-project/data/labels_BinClass.csv"
clinical_data = pd.read_csv(clinical_data_path)
clinical_data.dropna(inplace=True)
clinical_data.reset_index(drop=True, inplace=True)
clinical_data['w8_responder'] = clinical_data['w8_responder'].map({'Yes': 1, 'No': 0})
clinical_data['Stage1TX'] = clinical_data['Stage1TX'].map({'SER': 1, 'PLA': 0})
subjects_to_remove = ['CU0058', 'CU0059', 'CU0060', 'CU0061', 'CU0068', 'CU0074']
filtered_clinical_data = clinical_data[~clinical_data['ProjectSpecificId'].isin(subjects_to_remove)]
filtered_clinical_data.reset_index(drop=True, inplace=True)
sertraline_df = filtered_clinical_data[filtered_clinical_data['Stage1TX'] == 1]


fmri_data_path = "/data/projects/depredict/repositories/EMBARC/data/data_bids/derivatives/_fmriprep/output/"
subject_ids = sertraline_df['ProjectSpecificId'].tolist()
fmri_files = {}
missing_subjects = []
for sub_id in subject_ids:
    file_pattern = os.path.join(fmri_data_path,
                                f"sub-{sub_id}/ses-1/func/sub-{sub_id}_ses-1_task-rest1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz")
    if os.path.exists(file_pattern):
        fmri_files[sub_id] = file_pattern
    else:
        missing_subjects.append(sub_id)
processed_df = sertraline_df[sertraline_df['ProjectSpecificId'].isin(fmri_files.keys())]
processed_df = processed_df.copy()
processed_df['fMRI_path'] = processed_df['ProjectSpecificId'].map(fmri_files)
processed_df
# </editor-fold>

In [ ]:

# <editor-fold desc="Data Classes & Functions">
class fMRIDataset(Dataset):
    def __init__(self, df, data_dict, transform=None):
        """
        df: DataFrame with columns: ['subject_part', 'label']
        data_dict: {subject_part_id: 3D numpy array or torch.Tensor}
        """
        self.data = df.reset_index(drop=True)
        self.data_dict = data_dict
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        subject_part = row['subject_part']
        label = row['label']

        image = self.data_dict[subject_part]  # Already precomputed

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

def preload_mean_images(df, mean_image_path):
    data_dict = {}
    part_paths = []

    # Step 1: Build list of expected files (both parts)
    for subject_id in df["ProjectSpecificId"]:
        part_paths.append((f"{subject_id}_part1.pt", f"{subject_id}_part1"))
        part_paths.append((f"{subject_id}_part2.pt", f"{subject_id}_part2"))

    # Step 2: Load each .pt file individually
    for filename, part_id in tqdm(part_paths, desc="Preloading mean images"):
        full_path = os.path.join(mean_image_path, filename)
        if not os.path.exists(full_path):
            print(f"⚠️ Missing: {filename}")
            continue

        tensor = torch.load(full_path).float()
        data_dict[part_id] = tensor

    return data_dict

# </editor-fold>


In [ ]:
def precompute_mean_images(data_dict, save_dir="/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/T_90/"):
    os.makedirs(save_dir, exist_ok=True)
    for subject_id, fmri_tensor in tqdm(data_dict.items(), desc="Precomputing mean images"):
        # Ensure the tensor has at least 180 timepoints
        if fmri_tensor.shape[0] < 180:
            print(f"Skipping {subject_id}: Not enough timepoints ({fmri_tensor.shape[0]}).")
            continue

        # Split into first 90 and last 90 timepoints
        first_90 = fmri_tensor[:90]  # Shape: [90, D, H, W]
        last_90 = fmri_tensor[-90:]  # Shape: [90, D, H, W]

        # Compute mean images
        mean_image_first = first_90.mean(dim=0)  # Shape: [D, H, W]
        mean_image_last = last_90.mean(dim=0)  # Shape: [D, H, W]

        # Normalize the mean images
        eps = 1e-6
        mean_image_first = (mean_image_first - mean_image_first.mean()) / (mean_image_first.std() + eps)
        mean_image_last = (mean_image_last - mean_image_last.mean()) / (mean_image_last.std() + eps)

        # Add a channel dimension to match model input shape
        mean_image_first = mean_image_first.unsqueeze(0)  # Shape: [1, D, H, W]
        mean_image_last = mean_image_last.unsqueeze(0)  # Shape: [1, D, H, W]

        # Save the mean images
        save_path_first = os.path.join(save_dir, f"{subject_id}_part1.pt")
        save_path_last = os.path.join(save_dir, f"{subject_id}_part2.pt")
        torch.save(mean_image_first, save_path_first)
        torch.save(mean_image_last, save_path_last)

        print(f"Saved mean images for {subject_id} at {save_path_first} and {save_path_last}")

In [ ]:

# <editor-fold desc="Model Definition">
class Simple3DCNN(nn.Module):
    def __init__(self):
        super(Simple3DCNN, self).__init__()
        self.conv1 = nn.Conv3d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1)

        # Use Global Average Pooling to reduce feature map size efficiently
        self.global_pool = nn.AdaptiveAvgPool3d((4, 4, 4))

        # FC Layer adjusted for the reduced size
        self.fc1 = nn.Linear(64 * 4 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        # Reduce feature map size while retaining granularity
        x = self.global_pool(x)

        x = x.view(x.size(0), -1)  # Flatten for FC layer
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class HighAccuracy3DCNN(nn.Module):
    def __init__(self):
        super(HighAccuracy3DCNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm3d(8),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv3d(8, 16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.feature_dim = 64 * 6 * 7 * 6

        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(self.feature_dim, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.conv1(x)  # [B, 8, ~48, ~57, ~48]
        x = self.conv2(x)  # [B, 16, ~24, ~28, ~24]
        x = self.conv3(x)  # [B, 32, ~12, ~14, ~12]
        x = self.conv4(x)  # [B, 64, ~6, ~7, ~6]

        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)  # No sigmoid here — use BCEWithLogitsLoss

        return x
# </editor-fold>


In [ ]:

# <editor-fold desc="Training & Evaluation Loop">

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    warnings.filterwarnings("ignore", category=FutureWarning)

    history = {
        'epoch': [],
        'train_loss': [],
        'train_accuracy': [],
        'train_f1': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': []
    }

    total_start_time = time.time()

    for epoch in range(num_epochs):
        print(f"\n🚀 Starting Epoch {epoch + 1}/{num_epochs}")
        model.train()
        total_loss = 0
        all_preds = []
        all_labels = []

        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Train Epoch {epoch+1}'):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

            with torch.no_grad():
                preds = (torch.sigmoid(outputs) > 0.5).int()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Compute training metrics
        train_loss = total_loss / len(train_loader)
        train_accuracy = (np.array(all_preds) == np.array(all_labels)).mean()
        try:
            train_f1 = f1_score(all_labels, all_preds)
        except:
            train_f1 = 0.0

        # Validation
        val_loss, val_accuracy, val_precision, val_recall, val_f1 = evaluate_model(model, val_loader, criterion)

        # Store metrics
        history['epoch'].append(epoch + 1)
        history['train_loss'].append(train_loss)
        history['train_accuracy'].append(train_accuracy)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_accuracy)
        history['val_precision'].append(val_precision)
        history['val_recall'].append(val_recall)
        history['val_f1'].append(val_f1)

        # Log
        print(f"\n📊 Epoch {epoch + 1} Summary:")
        print(f"   Train Loss: {train_loss:.4f} | Accuracy: {train_accuracy*100:.2f}% | F1: {train_f1:.4f}")
        print(f"   Val Loss: {val_loss:.4f} | Accuracy: {val_accuracy*100:.2f}%")
        print(f"   Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f}")

    total_time = time.time() - total_start_time
    print(f"\n⏱️ Total Training Time: {total_time:.2f} seconds")

    return history

def evaluate_model(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels.float())
            total_loss += loss.item()

            predicted = (torch.sigmoid(outputs) > 0.5).int()
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = total_loss / len(val_loader)
    accuracy = (np.array(all_predictions) == np.array(all_labels)).mean()
    try:
        precision = precision_score(all_labels, all_predictions)
        recall = recall_score(all_labels, all_predictions)
        f1 = f1_score(all_labels, all_predictions)
    except:
        precision = recall = f1 = 0.0

    return avg_val_loss, accuracy, precision, recall, f1

# </editor-fold>


In [ ]:
# data_dict = {}
#
# for idx, row in tqdm(processed_df.iterrows(), total=len(processed_df), desc="Loading 4D fMRI data"):
#     sub_id = row["ProjectSpecificId"]
#     fmri_path = row["fMRI_path"]
#     try:
#         img = nib.load(fmri_path)
#         data = img.get_fdata()  # [X, Y, Z, T]
#         data_tensor = torch.tensor(data.transpose(3, 0, 1, 2))  # shape: [T, D, H, W]
#         data_dict[sub_id] = data_tensor
#     except Exception as e:
#         print(f"⚠️ Failed loading for {sub_id}: {e}")
#
# precompute_mean_images(data_dict)

In [ ]:

#<editor-fold desc="Non-Slurm_1 Execution">
# Path to precomputed mean .pt files
mean_image_path = "/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/T_90/"

# Load precomputed mean image tensors
data_dict = preload_mean_images(processed_df, mean_image_path)  # This now returns {subject_part: tensor}

mean_df = pd.DataFrame([
    {"subject_part": part_id,
     "label": int(processed_df.loc[processed_df["ProjectSpecificId"] == part_id.split("_part")[0], "w8_responder"].values[0])
    }
    for part_id in data_dict.keys()
])

mean_df
# </editor-fold>

In [ ]:

#<editor-fold desc="Non-Slurm_2 Execution">

# Stratified data split
train_data, test_data = train_test_split(mean_df, test_size=0.2, stratify=mean_df["label"], random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.25, stratify=train_data["label"], random_state=42)

# Prepare datasets and data loaders
train_dataset = fMRIDataset(train_data, data_dict=data_dict)
val_dataset = fMRIDataset(val_data, data_dict=data_dict)
test_dataset = fMRIDataset(test_data, data_dict=data_dict)

batch_size = 8  # Adjust based on your GPU
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=16, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model Preparation
model = HighAccuracy3DCNN()  # or your model of choice
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ Current Device: {torch.cuda.current_device()}")
print(f"✅ GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")

# </editor-fold>

In [ ]:
len(train_loader)  # Number of batches in the training set

In [ ]:
# Training & Evaluation
history = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5)

In [ ]:

# <editor-fold desc="Epoch Plots">

def plot_metrics(history, model=None, test_loader=None, device=None, save_dir=None):
    """
    Plots training metrics and optionally evaluation metrics on the test set.

    Parameters:
        history (dict): Output from train_model function.
        model (torch.nn.Module): Trained model for evaluation (optional).
        test_loader (DataLoader): Test set for confusion matrix and ROC (optional).
        device (torch.device): Device to run inference on (optional).
        save_dir (str): Directory to save plots (optional).
    """

    def _savefig(name):
        if save_dir:
            plt.savefig(f"{save_dir}/{name}.png", dpi=300, bbox_inches="tight")

    # Plot 1: Train vs Val Loss
    plt.figure(figsize=(7, 5))
    plt.plot(history['epoch'], history['train_loss'], label='Train Loss')
    plt.plot(history['epoch'], history['val_loss'], label='Validation Loss')
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    _savefig("loss_curve")
    plt.show()

    # Plot 2: Validation Accuracy
    plt.figure(figsize=(7, 5))
    plt.plot(history['epoch'], [v * 100 for v in history['val_accuracy']], label='Validation Accuracy (%)')
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title("Validation Accuracy Over Time")
    plt.grid(True)
    _savefig("val_accuracy")
    plt.show()

    # Plot 3: Precision, Recall, F1
    plt.figure(figsize=(7, 5))
    plt.plot(history['epoch'], history['val_precision'], label='Precision')
    plt.plot(history['epoch'], history['val_recall'], label='Recall')
    plt.plot(history['epoch'], history['val_f1'], label='F1 Score')
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Validation Precision, Recall, and F1")
    plt.legend()
    plt.grid(True)
    _savefig("val_metrics")
    plt.show()

    # Optional: Confusion Matrix and ROC Curve
    if model and test_loader and device:
        print("📊 Evaluating on test set for confusion matrix and ROC curve...")
        model.eval()
        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images).squeeze(1)
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).int()

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        # Confusion Matrix
        cm = confusion_matrix(all_labels, all_preds)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-responder", "Responder"])
        disp.plot(cmap="Blues")
        plt.title("Confusion Matrix (Test Set)")
        _savefig("confusion_matrix")
        plt.show()

        # ROC Curve
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        roc_auc = auc(fpr, tpr)

        plt.figure(figsize=(7, 5))
        plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve (Test Set)")
        plt.legend()
        plt.grid(True)
        _savefig("roc_curve")
        plt.show()

# </editor-fold>

In [ ]:

plot_metrics(history, model=model, test_loader=test_loader, device=device)
